# Remove the unsupported charge-subcategory attribute
Manual maintenance only, on either compute scenario. Pause jobs and dashboard access during this migration; run DEV first, inspect the controls, then run PROD. Update the repository before running this notebook.
The source has no ChargeSubcategory. This notebook removes the constant attribute from dim_charge_type and rebuilds only dm_cost_by_charge_type. Existing charge_type_sk values are preserved, including their legacy hash compatibility token. Facts, source Parquet, RAW, Bronze, Silver and security entitlements are not written. No full reload is needed.
Run all cells in the same Spark notebook session, not as a SQL Warehouse task. Set the confirmation explicitly. The previous Delta version is printed for a manual RESTORE if necessary. If the datamart refresh fails after the dimension migration, stop jobs and rerun this notebook after correcting the error; do not resume loading with mismatched schemas.

In [ ]:
from pathlib import Path
import sys

source_root = next((root / 'src' for root in (Path.cwd(), *Path.cwd().parents) if (root / 'src').is_dir()), None)
if source_root is not None and str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))
if source_root is None:
    raise ValueError('Open this notebook from the updated project Git folder.')

In [ ]:
ENVIRONMENT = 'dev'
CONFIRMATION = ''

if ENVIRONMENT not in {'dev', 'prod'}:
    raise ValueError('environment must be dev or prod')
if CONFIRMATION != 'REMOVE_CHARGE_SUBCATEGORY':
    raise ValueError('Read the maintenance instructions, then set CONFIRMATION = REMOVE_CHARGE_SUBCATEGORY.')

In [ ]:
import hashlib

def validate_charge_type_projection(rows):
    # Keep the old hash format; never reinterpret a business value as a salt.
    result, keys = [], set()
    for row in rows:
        if row.get('charge_subcategory') not in {None, 'Unknown'}:
            raise ValueError('STOP: charge_subcategory contains a real value; migration refused.')
        key = row['charge_type_sk']
        parts = ('charge_type', row['charge_category'], 'Unknown', row['charge_frequency'])
        expected = hashlib.sha256('||'.join(value for value in parts if value is not None).encode()).hexdigest()
        if key != expected or key in keys:
            raise ValueError('STOP: unexpected or duplicate charge_type_sk; migration refused.')
        keys.add(key)
        result.append({name: row[name] for name in ('charge_type_sk', 'charge_category', 'charge_frequency')})
    return result

In [ ]:
from uuid import uuid4
from finops_cloud.config import load_config
from finops_cloud.sql.runner import execute_sql_file, sql_text, table_context

config = load_config(ENVIRONMENT)
if config.catalog != f'finops_{ENVIRONMENT}':
    raise ValueError('STOP: catalog override does not match the selected environment.')
if 'charge_subcategory' in sql_text('datamarts/table_refresh/06_dm_cost_by_charge_type.sql').lower():
    raise ValueError('STOP: loading SQL is still the old version; update the Git folder first.')
dimension = config.table('dim_charge_type', 'gold')
fact = config.table('fact_cost_usage', 'gold')
datamart = config.table('dm_cost_by_charge_type', 'datamart')
frame = spark.table(dimension)
if set(frame.columns) - {'charge_type_sk', 'charge_category', 'charge_frequency', 'charge_subcategory'}:
    raise ValueError('STOP: dimension has additional attributes; inspect them before migrating.')
raw_rows = frame.limit(10001).collect()
if len(raw_rows) > 10000:
    raise ValueError('STOP: unexpectedly large charge-type dimension; inspect it before migrating.')
projected = validate_charge_type_projection([row.asDict() for row in raw_rows])

control_sql = f'''SELECT count(*) AS fact_rows, sum(f.billed_cost) AS billed_cost,
    sum(f.effective_cost) AS effective_cost,
    coalesce(sum(CASE WHEN d.charge_type_sk IS NULL THEN 1 ELSE 0 END), 0) AS orphan_rows
    FROM {fact} f LEFT JOIN {dimension} d ON f.charge_type_sk = d.charge_type_sk'''
before = spark.sql(control_sql).first().asDict()
if before['orphan_rows'] != 0:
    raise ValueError('STOP: facts already have orphan charge-type keys; migration refused.')

if 'charge_subcategory' in frame.columns:
    previous_version = spark.sql(f'DESCRIBE HISTORY {dimension}').select('version').first()[0]
    print(f'Previous dimension version: {previous_version}. Manual recovery: RESTORE TABLE {dimension} TO VERSION AS OF {previous_version};')
    schema = frame.select('charge_type_sk', 'charge_category', 'charge_frequency').schema
    # Rows are materialized on the driver: replacement does not read its own target.
    snapshot = spark.createDataFrame(projected, schema)
    temporary_view = f'migration_charge_type_{uuid4().hex}'
    snapshot.createOrReplaceTempView(temporary_view)
    try:
        spark.sql(f'CREATE OR REPLACE TABLE {dimension} USING DELTA AS SELECT * FROM {temporary_view}').collect()
        spark.sql(f'ALTER TABLE {dimension} ALTER COLUMN charge_type_sk SET NOT NULL').collect()
    finally:
        spark.catalog.dropTempView(temporary_view)
else:
    print('Dimension already migrated; rechecking it and refreshing the datamart.')

after_rows = [row.asDict() for row in spark.table(dimension).collect()]
if sorted(after_rows, key=lambda row: row['charge_type_sk']) != sorted(projected, key=lambda row: row['charge_type_sk']):
    raise ValueError('STOP: dimension keys changed; keep jobs paused and inspect the printed recovery version.')
if 'charge_subcategory' in spark.table(dimension).columns:
    raise ValueError('STOP: unsupported attribute still exists.')
after = spark.sql(control_sql).first().asDict()
if after != before:
    raise ValueError('STOP: fact counts, costs or relationships changed; keep jobs paused.')

execute_sql_file(spark, 'datamarts/table_refresh/06_dm_cost_by_charge_type.sql', table_context(config))
if 'charge_subcategory' in spark.table(datamart).columns:
    raise ValueError('STOP: datamart still uses the old schema; update the project SQL and rerun.')
mart_totals = spark.sql(f'SELECT coalesce(sum(total_billed_cost), 0) AS billed_cost FROM {datamart}').first()[0]
if mart_totals != (before['billed_cost'] or 0):
    raise ValueError('STOP: rebuilt datamart does not reconcile with the facts.')
print(f'PASS: {ENVIRONMENT}; unsupported attribute removed; keys, fact rows and costs preserved. Run the updated script 04 manually in PROD, then 05.')